In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("../Ontrac_data.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Clean text columns
df["Products"] = df["Products"].str.strip()
df["Bank_Document"] = df["Bank_Document"].str.strip()

# Clean product naming inconsistencies
df["Products"] = (
    df["Products"]
    .str.replace("WHITE + GREY", "WHITE+GREY", regex=False)
)

# Convert Amount to numeric
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# Remove missing values
df = df.dropna(
    subset=["Products", "Bank_Document", "Amount"]
)

# Visualization style
sns.set_theme(style="whitegrid")
df

,Invoice_Date,Client,Products,Amount,Bank_Document
0,9/2/2025,NOVAQUA,SIKACERAM 80I WHITE+GREY,21420.00,TRANSFER RECEIVED
1,9/8/2025,MY DE CONSTRUCTION,REBAR 08 TOR FE 500,11144.00,COLLECTION OF 1 BILL OF EXCHANGE
2,9/17/2025,TERRASTRUCT,CEMENT CPJ35 (21 T +19 bags),36790.00,TRANSFER RECEIVED
3,9/22/2025,STE DERRAB LISSAKAN,SIKACERAM 80I WHITE+GREY,4998.68,OFF-BRANCH CASH DEPOSIT
4,9/23/2025,STE DERRAB LISSAKAN,WOOD TALK LOUVERS 200X1200,4999.60,OFF-BRANCH CASH DEPOSIT
...,...,...,...,...,...
142,8/24/2026,AYOUB MARRAKECH,WEBERCOL CLASSIC GREY,4254.48,OFF-BRANCH CASH DEPOSIT
143,8/24/2026,AYOUB MARRAKECH,WEBERCOL CLASSIC GREY,3999.40,OFF-BRANCH CASH DEPOSIT
144,8/24/2026,AYOUB MARRAKECH,WEBERCOL CLASSIC GREY,3999.40,OFF-BRANCH CASH DEPOSIT
145,8/28/2026,ISMANE DEVELOPPEMENT,WEBERCOL CLASSIC GREY,3916.00,OFF-BRANCH CASH DEPOSIT


## 1. Which payment methods are most commonly used for each product?

In [4]:
payment_frequency = pd.crosstab(
    df["Products"],
    df["Bank_Document"]
)

payment_frequency_summary = pd.DataFrame({
    "Most_Common_Payment": payment_frequency.idxmax(axis=1),
    "Transaction_Count": payment_frequency.max(axis=1)
})
payment_frequency_summary

,Most_Common_Payment,Transaction_Count
Products,,
CEMENT CPJ35,TRANSFER RECEIVED,1
CEMENT CPJ35 (21 T +19 bags),TRANSFER RECEIVED,1
CEMENT CPJ35 (27 T),TRANSFER RECEIVED,8
CEMENT CPJ35 (27 T)+ PROLIDAL,TRANSFER RECEIVED,4
CEMENT CPJ35 (27 T)+SIKACERAM 80I WHITE,TRANSFER RECEIVED,1
CEMENT CPJ35 (28 T),TRANSFER RECEIVED,5
CEMENT CPJ35 (29 T),TRANSFER RECEIVED,2
CEMENT CPJ35+ REBAR 12 TOR,TRANSFER RECEIVED,1
CEMENT CPJ35+REBAR 12 TOR,TRANSFER RECEIVED,1


## 2. Are expensive products more likely to be paid for by bank transfer?

In [8]:
payment_average = (
    df.groupby("Bank_Document")["Amount"]
      .agg(
          Average_Amount="mean",
          Total_Amount="sum",
          Transaction_Count="count"
      )
      .sort_values("Average_Amount", ascending=False)
)

payment_average["Percentage"] = (
    payment_average["Total_Amount"]
    / payment_average["Total_Amount"].sum()
    * 100
)

payment_average

,Average_Amount,Total_Amount,Transaction_Count,Percentage
Bank_Document,,,,
TRANSFER RECEIVED,34921.241379,2025432.00,58,76.980957
COLLECTION OF 1 LCN,28879.333333,173276.00,6,6.585732
CHEQUE DEPOSITED FOR COLLECTION,24781.666667,74345.00,3,2.825644
COLLECTION OF 2 LCN,18000.000000,18000.00,1,0.684129
COLLECTION OF 1 BILL OF EXCHANGE,11144.000000,11144.00,1,0.423552
OFF-BRANCH CASH DEPOSIT,4216.472821,328884.88,78,12.499987


In [ ]:
## the percentage of high-value invoices per category of Bank_Document. 
## A high-value invoice is defined as an invoice with an amount greater than or equal to the median amount of all invoices.

high_value_threshold = df["Amount"].median()

df["High_Value"] = df["Amount"] >= high_value_threshold

high_value_payment = pd.crosstab(
    df["High_Value"],
    df["Bank_Document"],
    normalize="index"
) * 100

high_value_payment.round(2)


Bank_Document,CHEQUE DEPOSITED FOR COLLECTION,COLLECTION OF 1 BILL OF EXCHANGE,COLLECTION OF 1 LCN,COLLECTION OF 2 LCN,OFF-BRANCH CASH DEPOSIT,TRANSFER RECEIVED
High_Value,,,,,,
False,0.00,0.00,0.00,0.00,100.00,0.00
True,3.85,1.28,7.69,1.28,11.54,74.36


## 3. Are certain products disproportionately associated with cash deposits?

In [12]:
cash_df = df[
    df["Bank_Document"] == "OFF-BRANCH CASH DEPOSIT"
]
#Calculate cash-deposit frequency by product:

cash_by_product = (
    cash_df.groupby("Products")
           .size()
           .sort_values(ascending=False)
)

cash_by_product

Products
WEBERCOL CLASSIC WHITE                  14
WEBERCOL CLASSIC GREY                    9
SIKACERAM 80I+HAMRI ADHESIVE             8
SIKACERAM 80I WHITE                      7
SIKACERAM 80I WHITE+GREY                 5
SIKACERAM 100M+80I                       5
GURU WATERSTOP MAT (30 m2)               4
SIKACERAM 100M GREY                      4
HAMRI ADHESIVE GREY C1                   2
WEBERCOL DUR WHITE                       2
HAMRI ADHESIVE GREY C1 25KG              2
WEBERCOL CLASSIC+JUNTA FINA              2
SIKACERAM 100M WHITE                     2
HAMRI ADHESIVE GREY C1 + WHITE C1        1
IMPERFLEX                                1
HAMRI ADHESIVE WHITE C1 + C2             1
HAMRI ADHESIVE WHITE C1                  1
REBAR 05 SMOOTH                          1
SIKACERAM 100M GREY + WHITE              1
WEBERCOL CLASSIC GREY+DUR WHITE          1
WEBERCOL CLASSIC GREY+PROLIDAL           1
WEBERCOL CLASSIC WHITE+GREY              1
WEBERCOL CLASSIC GREY+WHITE              1
WE

In [14]:
## Percentage of each product's transactions that are cash deposits

total_by_product = df.groupby("Products").size()

cash_percentage = (
    cash_by_product / total_by_product * 100
).sort_values(ascending=False)

cash_percentage

Products
GURU WATERSTOP MAT (30 m2)                 100.000000
HAMRI ADHESIVE GREY C1                     100.000000
HAMRI ADHESIVE GREY C1 + WHITE C1          100.000000
HAMRI ADHESIVE GREY C1 25KG                100.000000
HAMRI ADHESIVE WHITE C1                    100.000000
HAMRI ADHESIVE WHITE C1 + C2               100.000000
IMPERFLEX                                  100.000000
SIKACERAM 100M GREY                        100.000000
SIKACERAM 100M GREY + WHITE                100.000000
WOOD TALK LOUVERS 200X1200                 100.000000
SIKACERAM 100M WHITE                       100.000000
SIKACERAM 80I WHITE                        100.000000
WEBERCOL CLASSIC GREY+DUR WHITE            100.000000
WEBERCOL CLASSIC GREY                      100.000000
WEBERCOL DUR+JUNTA FINA+CLASSIC GREY       100.000000
WEBERCOL CLASSIC WHITE+GREY                100.000000
WEBERCOL CLASSIC GREY+PROLIDAL             100.000000
WEBERCOL DUR WHITE                         100.000000
SIKACERAM 80I+HAMRI

## 4. Which products have the highest amount collected through each payment method?

In [15]:
amount_by_payment = pd.pivot_table(
    df,
    values="Amount",
    index="Products",
    columns="Bank_Document",
    aggfunc="sum",
    fill_value=0
)

amount_by_payment

Bank_Document,CHEQUE DEPOSITED FOR COLLECTION,COLLECTION OF 1 BILL OF EXCHANGE,COLLECTION OF 1 LCN,COLLECTION OF 2 LCN,OFF-BRANCH CASH DEPOSIT,TRANSFER RECEIVED
Products,,,,,,
CEMENT CPJ35,0.0,0.0,0.0,0.0,0.00,40000.0
CEMENT CPJ35 (21 T +19 bags),0.0,0.0,0.0,0.0,0.00,36790.0
CEMENT CPJ35 (27 T),0.0,0.0,0.0,0.0,0.00,322380.0
CEMENT CPJ35 (27 T)+ PROLIDAL,0.0,0.0,0.0,0.0,0.00,160920.0
CEMENT CPJ35 (27 T)+SIKACERAM 80I WHITE,0.0,0.0,0.0,0.0,0.00,41720.0
CEMENT CPJ35 (28 T),0.0,0.0,0.0,0.0,0.00,199230.0
CEMENT CPJ35 (29 T),0.0,0.0,0.0,0.0,0.00,86420.0
CEMENT CPJ35+ REBAR 12 TOR,0.0,0.0,0.0,0.0,0.00,41600.0
CEMENT CPJ35+REBAR 12 TOR,0.0,0.0,0.0,0.0,0.00,41445.0


In [16]:
## Find the highest product for each payment method

highest_product_by_payment = amount_by_payment.idxmax()

highest_amount_by_payment = amount_by_payment.max()

payment_leaders = pd.DataFrame({
    "Top_Product": highest_product_by_payment,
    "Total_Amount": highest_amount_by_payment
})

payment_leaders

,Top_Product,Total_Amount
Bank_Document,,
CHEQUE DEPOSITED FOR COLLECTION,SIKACERAM 80I GREY + PROLIDAL,54185.00
COLLECTION OF 1 BILL OF EXCHANGE,REBAR 08 TOR FE 500,11144.00
COLLECTION OF 1 LCN,SIKACERAM 80I WHITE+GREY+REBAR 10 TOR,45600.00
COLLECTION OF 2 LCN,WEBERCOL CLASSIC+JUNTA FINA,18000.00
OFF-BRANCH CASH DEPOSIT,WEBERCOL CLASSIC WHITE,60484.97
TRANSFER RECEIVED,CIMARPRO 35 (27 T),447555.00


## 5. Which payment method generates the highest total collection?

In [18]:
payment_total = (
    df.groupby("Bank_Document")["Amount"]
      .sum()
      .sort_values(ascending=False)
)


#Percentage contribution
payment_total_percentage = (
    payment_total / df["Amount"].sum()
) * 100

payment_total_percentage.round(2)

Bank_Document
TRANSFER RECEIVED                   76.98
OFF-BRANCH CASH DEPOSIT             12.50
COLLECTION OF 1 LCN                  6.59
CHEQUE DEPOSITED FOR COLLECTION      2.83
COLLECTION OF 2 LCN                  0.68
COLLECTION OF 1 BILL OF EXCHANGE     0.42
Name: Amount, dtype: float64

## 6. Which products depend heavily on a single payment method?

In [19]:
payment_mix = pd.crosstab(
    df["Products"],
    df["Bank_Document"],
    normalize="index"
) * 100


dominant_payment = payment_mix.idxmax(axis=1)

dominant_percentage = payment_mix.max(axis=1)

payment_dependency = pd.DataFrame({
    "Dominant_Payment_Method": dominant_payment,
    "Dominant_Payment_Percentage": dominant_percentage
})

payment_dependency = payment_dependency.sort_values(
    "Dominant_Payment_Percentage",
    ascending=False
)

payment_dependency

,Dominant_Payment_Method,Dominant_Payment_Percentage
Products,,
CEMENT CPJ35,TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (21 T +19 bags),TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (27 T),TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (27 T)+ PROLIDAL,TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (27 T)+SIKACERAM 80I WHITE,TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (28 T),TRANSFER RECEIVED,100.000000
CEMENT CPJ35 (29 T),TRANSFER RECEIVED,100.000000
CEMENT CPJ35+ REBAR 12 TOR,TRANSFER RECEIVED,100.000000
CEMENT CPJ35+REBAR 12 TOR,TRANSFER RECEIVED,100.000000


## 7. Do different payment methods have different transaction sizes?

In [20]:
payment_statistics = (
    df.groupby("Bank_Document")["Amount"]
      .agg(
          Transaction_Count="count",
          Total_Amount="sum",
          Average_Amount="mean",
          Median_Amount="median",
          Minimum_Amount="min",
          Maximum_Amount="max",
          Std_Dev="std"
      )
      .sort_values(
          "Total_Amount",
          ascending=False
      )
)

payment_statistics

,Transaction_Count,Total_Amount,Average_Amount,Median_Amount,Minimum_Amount,Maximum_Amount,Std_Dev
Bank_Document,,,,,,,
TRANSFER RECEIVED,58,2025432.00,34921.241379,40230.00,6000.0,48500.0,10845.748932
OFF-BRANCH CASH DEPOSIT,78,328884.88,4216.472821,4550.72,384.0,4999.8,1009.730867
COLLECTION OF 1 LCN,6,173276.00,28879.333333,27938.00,12500.0,45600.0,14341.040502
CHEQUE DEPOSITED FOR COLLECTION,3,74345.00,24781.666667,22560.00,20160.0,31625.0,6046.768421
COLLECTION OF 2 LCN,1,18000.00,18000.000000,18000.00,18000.0,18000.0,NaN
COLLECTION OF 1 BILL OF EXCHANGE,1,11144.00,11144.000000,11144.00,11144.0,11144.0,NaN


## 8. Which product-payment combinations generate the most revenue?

In [21]:
product_payment_revenue = (
    df.groupby(
        ["Products", "Bank_Document"]
    )["Amount"]
    .sum()
    .sort_values(ascending=False)
)


product_payment_summary = (
    product_payment_revenue
    .reset_index(name="Total_Amount")
)

product_payment_summary

,Products,Bank_Document,Total_Amount
0,CIMARPRO 35 (27 T),TRANSFER RECEIVED,447555.00
1,CEMENT CPJ35 (27 T),TRANSFER RECEIVED,322380.00
2,CEMENT CPJ35 (28 T),TRANSFER RECEIVED,199230.00
3,CEMENT CPJ35 (27 T)+ PROLIDAL,TRANSFER RECEIVED,160920.00
4,CEMENT CPJ35 (29 T),TRANSFER RECEIVED,86420.00
...,...,...,...
59,IMPERFLEX,OFF-BRANCH CASH DEPOSIT,4050.00
60,WEBERCOL CLASSIC GREY+DUR WHITE,OFF-BRANCH CASH DEPOSIT,2964.38
61,REBAR 05 SMOOTH,OFF-BRANCH CASH DEPOSIT,2800.00
62,WEBERCOL CLASSIC GREY+WHITE,OFF-BRANCH CASH DEPOSIT,2725.00
